In [6]:
# Q3 - Step 1: Personal parameters, reproducibility, and GPU setup

import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import time

# --------------------------------------------------
# Personal Parameters
# --------------------------------------------------

SID4 = 1980
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6

CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10

print("Personal Parameters")
print("-------------------")
print("SID4  =", SID4)
print("SEED  =", SEED)
print("SLICE =", SLICE)
print("HP_ID =", HP_ID)
print("CLS_A =", CLS_A)
print("CLS_B =", CLS_B)


# --------------------------------------------------
# Reproducibility
# --------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# --------------------------------------------------
# GPU Setup
# --------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nHardware")
print("--------")
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not enabled.")

Personal Parameters
-------------------
SID4  = 1980
SEED  = 1980
SLICE = 980
HP_ID = 0
CLS_A = 0
CLS_B = 1

Hardware
--------
Device: cuda
GPU: Tesla T4


In [7]:
# Q3 - Step 2: Create dataset and define fixed experiment settings

# --------------------------------------------------
# Fixed experiment settings
# These values will remain the SAME for all tests
# so that the optimization comparisons are fair.
# --------------------------------------------------

INPUT_SIZE = 128       # Number of input features
HIDDEN_SIZE = 256      # Number of neurons in the hidden layer
OUTPUT_SIZE = 10       # Number of output classes

BATCH_SIZE = 64        # Number of samples processed per batch
TRAINING_STEPS = 100   # Number of training steps
NUM_SAMPLES = 6400     # Total number of synthetic samples


# --------------------------------------------------
# Reset seed before creating the dataset
# This makes the generated data reproducible.
# --------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# --------------------------------------------------
# Create synthetic classification dataset
# X = input features
# y = class labels from 0 to 9
# Keep the master copy on CPU; each experiment will
# control when/how data is moved to the GPU.
# --------------------------------------------------

X = torch.randn(
    NUM_SAMPLES,
    INPUT_SIZE
)

y = torch.randint(
    low=0,
    high=OUTPUT_SIZE,
    size=(NUM_SAMPLES,)
)


# --------------------------------------------------
# Verify the experiment configuration
# --------------------------------------------------

print("Experiment Configuration")
print("------------------------")
print("Input size:", INPUT_SIZE)
print("Hidden size:", HIDDEN_SIZE)
print("Output classes:", OUTPUT_SIZE)
print("Number of samples:", NUM_SAMPLES)
print("Batch size:", BATCH_SIZE)
print("Training steps:", TRAINING_STEPS)

print("\nDataset")
print("-------")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("X device:", X.device)
print("y device:", y.device)

Experiment Configuration
------------------------
Input size: 128
Hidden size: 256
Output classes: 10
Number of samples: 6400
Batch size: 64
Training steps: 100

Dataset
-------
X shape: torch.Size([6400, 128])
y shape: torch.Size([6400])
X device: cpu
y device: cpu


In [8]:
# Q3 - Step 3: Define the common model and baseline training function

# --------------------------------------------------
# Define a simple neural network
# The same model architecture will be used for all
# optimization experiments.
# --------------------------------------------------

class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()

        # First fully connected layer
        self.fc1 = nn.Linear(INPUT_SIZE, HIDDEN_SIZE)

        # ReLU activation
        self.relu = nn.ReLU()

        # Output layer
        self.fc2 = nn.Linear(HIDDEN_SIZE, OUTPUT_SIZE)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)

        return x


# --------------------------------------------------
# Baseline training function
# --------------------------------------------------

def train_model(model, X, y):

    # Move the model to GPU
    model = model.to(device)

    # Loss function for multi-class classification
    criterion = nn.CrossEntropyLoss()

    # Adam optimizer
    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001
    )

    # Reset GPU peak-memory statistics
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    # Synchronize GPU before timing
    torch.cuda.synchronize()

    start_time = time.time()

    model.train()

    # --------------------------------------------------
    # Run the fixed number of training steps
    # --------------------------------------------------
    for step in range(TRAINING_STEPS):

        # Select a batch from the CPU dataset
        start = (step * BATCH_SIZE) % NUM_SAMPLES
        end = start + BATCH_SIZE

        batch_X = X[start:end]
        batch_y = y[start:end]

        # Transfer the current batch to GPU
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        # Clear old gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(batch_X)

        # Calculate loss
        loss = criterion(outputs, batch_y)

        # Backpropagation
        loss.backward()

        # Update model weights
        optimizer.step()

    # Wait until GPU operations finish before stopping timer
    torch.cuda.synchronize()

    training_time = time.time() - start_time

    # --------------------------------------------------
    # Measure peak GPU memory
    # --------------------------------------------------
    if torch.cuda.is_available():
        peak_memory = (
            torch.cuda.max_memory_allocated()
            / (1024 ** 2)
        )
    else:
        peak_memory = 0

    # Return measured values
    return training_time, peak_memory, loss.item()


print("Model and training function defined successfully!")

Model and training function defined successfully!


In [9]:
# Q3 - Step 4: Run baseline training experiment

# --------------------------------------------------
# Reset the seed before model initialization
# This ensures reproducible weight initialization.
# --------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


# --------------------------------------------------
# Create the baseline model
# PyTorch's default weight initialization is used.
# --------------------------------------------------

baseline_model = SimpleNet()


# --------------------------------------------------
# Train the baseline model
# --------------------------------------------------

baseline_time, baseline_memory, baseline_loss = train_model(
    baseline_model,
    X,
    y
)


# --------------------------------------------------
# Display baseline measurements
# --------------------------------------------------

print("Baseline Results")
print("----------------")

print(f"Training Time: {baseline_time:.4f} seconds")
print(f"Peak GPU Memory: {baseline_memory:.2f} MB")
print(f"Final Loss: {baseline_loss:.4f}")


Baseline Results
----------------
Training Time: 0.1287 seconds
Peak GPU Memory: 18.03 MB
Final Loss: 2.2875


In [10]:
# Q3 - Step 5: Compare CPU-to-GPU transfer vs direct GPU tensor creation

REPEATS = 5
ITERATIONS = 1000

cpu_to_gpu_times = []
direct_gpu_times = []


# --------------------------------------------------
# GPU warm-up
# This avoids measuring initial CUDA startup overhead.
# --------------------------------------------------

for _ in range(100):
    warmup_tensor = torch.randn(
        BATCH_SIZE,
        INPUT_SIZE,
        device=device
    )

torch.cuda.synchronize()


# --------------------------------------------------
# Method 1:
# Create tensor on CPU and then transfer it to GPU
# --------------------------------------------------

for repeat in range(REPEATS):

    torch.cuda.synchronize()
    start_time = time.perf_counter()

    for _ in range(ITERATIONS):

        # Tensor is first created in CPU memory
        tensor_cpu = torch.randn(
            BATCH_SIZE,
            INPUT_SIZE
        )

        # Tensor is then transferred to GPU memory
        tensor_gpu = tensor_cpu.to(device)

    # Wait for all GPU operations to finish
    torch.cuda.synchronize()

    elapsed = time.perf_counter() - start_time
    cpu_to_gpu_times.append(elapsed)


# --------------------------------------------------
# Method 2:
# Create the tensor directly in GPU memory
# --------------------------------------------------

for repeat in range(REPEATS):

    torch.cuda.synchronize()
    start_time = time.perf_counter()

    for _ in range(ITERATIONS):

        # Tensor is created directly on the GPU
        tensor_gpu = torch.randn(
            BATCH_SIZE,
            INPUT_SIZE,
            device=device
        )

    torch.cuda.synchronize()

    elapsed = time.perf_counter() - start_time
    direct_gpu_times.append(elapsed)


# --------------------------------------------------
# Calculate average times across the 5 repetitions
# --------------------------------------------------

cpu_to_gpu_mean = np.mean(cpu_to_gpu_times)
direct_gpu_mean = np.mean(direct_gpu_times)

speedup = cpu_to_gpu_mean / direct_gpu_mean


# --------------------------------------------------
# Display results
# --------------------------------------------------

print("Tensor Creation Results")
print("-----------------------")

print(
    f"CPU creation + GPU transfer (mean): "
    f"{cpu_to_gpu_mean:.4f} seconds"
)

print(
    f"Direct GPU creation (mean):         "
    f"{direct_gpu_mean:.4f} seconds"
)

print(f"Direct GPU speedup: {speedup:.2f}x")

Tensor Creation Results
-----------------------
CPU creation + GPU transfer (mean): 0.0786 seconds
Direct GPU creation (mean):         0.0106 seconds
Direct GPU speedup: 7.39x


In [11]:
# Q3 - Step 6: Compare Default vs Xavier Weight Initialization

REPEATS = 3

default_times = []
default_memories = []
default_losses = []

xavier_times = []
xavier_memories = []
xavier_losses = []


# --------------------------------------------------
# Remove the old baseline model from GPU memory
# We keep its recorded result variables, but free
# the actual model before starting new measurements.
# --------------------------------------------------

if "baseline_model" in globals():
    del baseline_model

torch.cuda.empty_cache()


# --------------------------------------------------
# Experiment 1: PyTorch Default Initialization
# --------------------------------------------------

for repeat in range(REPEATS):

    # Use a reproducible seed for each repetition
    current_seed = SEED + repeat

    random.seed(current_seed)
    np.random.seed(current_seed)
    torch.manual_seed(current_seed)
    torch.cuda.manual_seed_all(current_seed)

    # Create model using PyTorch's default initialization
    model = SimpleNet()

    # Train and collect measurements
    training_time, peak_memory, final_loss = train_model(
        model,
        X,
        y
    )

    default_times.append(training_time)
    default_memories.append(peak_memory)
    default_losses.append(final_loss)

    # Free GPU memory before the next run
    del model
    torch.cuda.empty_cache()


# --------------------------------------------------
# Experiment 2: Xavier Initialization
# --------------------------------------------------

for repeat in range(REPEATS):

    # Use the SAME corresponding seed as above
    current_seed = SEED + repeat

    random.seed(current_seed)
    np.random.seed(current_seed)
    torch.manual_seed(current_seed)
    torch.cuda.manual_seed_all(current_seed)

    # Create the same model architecture
    model = SimpleNet()

    # Apply Xavier initialization to all Linear layers
    for layer in model.modules():
        if isinstance(layer, nn.Linear):

            # Xavier initialization for weights
            nn.init.xavier_uniform_(layer.weight)

            # Initialize biases to zero
            nn.init.zeros_(layer.bias)

    # Train and collect measurements
    training_time, peak_memory, final_loss = train_model(
        model,
        X,
        y
    )

    xavier_times.append(training_time)
    xavier_memories.append(peak_memory)
    xavier_losses.append(final_loss)

    # Free GPU memory before the next run
    del model
    torch.cuda.empty_cache()


# --------------------------------------------------
# Calculate mean values
# --------------------------------------------------

default_time_mean = np.mean(default_times)
default_memory_mean = np.mean(default_memories)
default_loss_mean = np.mean(default_losses)

xavier_time_mean = np.mean(xavier_times)
xavier_memory_mean = np.mean(xavier_memories)
xavier_loss_mean = np.mean(xavier_losses)


# --------------------------------------------------
# Display results
# --------------------------------------------------

print("Weight Initialization Results")
print("-----------------------------")

print("\nDefault Initialization")
print(f"Mean Training Time: {default_time_mean:.4f} seconds")
print(f"Mean Peak GPU Memory: {default_memory_mean:.2f} MB")
print(f"Mean Final Loss: {default_loss_mean:.4f}")

print("\nXavier Initialization")
print(f"Mean Training Time: {xavier_time_mean:.4f} seconds")
print(f"Mean Peak GPU Memory: {xavier_memory_mean:.2f} MB")
print(f"Mean Final Loss: {xavier_loss_mean:.4f}")

Weight Initialization Results
-----------------------------

Default Initialization
Mean Training Time: 0.1221 seconds
Mean Peak GPU Memory: 18.06 MB
Mean Final Loss: 2.2893

Xavier Initialization
Mean Training Time: 0.1215 seconds
Mean Peak GPU Memory: 18.06 MB
Mean Final Loss: 2.4267


In [12]:
# Q3 - Step 7: Activation Checkpointing experiment

from torch.utils.checkpoint import checkpoint

REPEATS = 3

checkpoint_times = []
checkpoint_memories = []
checkpoint_losses = []


# --------------------------------------------------
# Model with activation checkpointing
# --------------------------------------------------

class CheckpointNet(nn.Module):
    def __init__(self):
        super(CheckpointNet, self).__init__()

        self.fc1 = nn.Linear(INPUT_SIZE, HIDDEN_SIZE)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(HIDDEN_SIZE, OUTPUT_SIZE)

    # Function that will be checkpointed
    def hidden_block(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        return x

    def forward(self, x):

        # Recompute hidden activations during backward pass
        x = checkpoint(
            self.hidden_block,
            x,
            use_reentrant=False
        )

        x = self.fc2(x)

        return x


# --------------------------------------------------
# Run activation checkpointing experiment
# --------------------------------------------------

for repeat in range(REPEATS):

    # Reproducible seed for each repetition
    current_seed = SEED + repeat

    random.seed(current_seed)
    np.random.seed(current_seed)
    torch.manual_seed(current_seed)
    torch.cuda.manual_seed_all(current_seed)

    # Create checkpointed model
    model = CheckpointNet()

    # Train using the same data, batch size,
    # learning rate, and training steps
    training_time, peak_memory, final_loss = train_model(
        model,
        X,
        y
    )

    checkpoint_times.append(training_time)
    checkpoint_memories.append(peak_memory)
    checkpoint_losses.append(final_loss)

    # Free GPU memory before next run
    del model
    torch.cuda.empty_cache()


# --------------------------------------------------
# Calculate mean results
# --------------------------------------------------

checkpoint_time_mean = np.mean(checkpoint_times)
checkpoint_memory_mean = np.mean(checkpoint_memories)
checkpoint_loss_mean = np.mean(checkpoint_losses)


# --------------------------------------------------
# Display results
# --------------------------------------------------

print("Activation Checkpointing Results")
print("--------------------------------")

print(f"Mean Training Time: {checkpoint_time_mean:.4f} seconds")
print(f"Mean Peak GPU Memory: {checkpoint_memory_mean:.2f} MB")
print(f"Mean Final Loss: {checkpoint_loss_mean:.4f}")

Activation Checkpointing Results
--------------------------------
Mean Training Time: 0.2367 seconds
Mean Peak GPU Memory: 19.08 MB
Mean Final Loss: 2.2893


In [13]:
# Q3 - Step 8: Gradient Accumulation experiment

REPEATS = 3

# --------------------------------------------------
# Gradient accumulation settings
#
# Effective batch size stays 64:
# 16 samples × 4 accumulation steps = 64 samples
# --------------------------------------------------

MICRO_BATCH_SIZE = 16
ACCUMULATION_STEPS = BATCH_SIZE // MICRO_BATCH_SIZE

accum_times = []
accum_memories = []
accum_losses = []


# --------------------------------------------------
# Run the experiment 3 times
# --------------------------------------------------

for repeat in range(REPEATS):

    # Use reproducible seeds
    current_seed = SEED + repeat

    random.seed(current_seed)
    np.random.seed(current_seed)
    torch.manual_seed(current_seed)
    torch.cuda.manual_seed_all(current_seed)

    # Create the same model architecture
    model = SimpleNet().to(device)

    # Same loss function and optimizer as baseline
    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001
    )

    # Reset peak GPU memory measurement
    torch.cuda.reset_peak_memory_stats()

    # Synchronize GPU before timing
    torch.cuda.synchronize()
    start_time = time.perf_counter()

    model.train()

    # --------------------------------------------------
    # Keep the same 100 optimizer training steps
    # --------------------------------------------------

    for step in range(TRAINING_STEPS):

        # Clear gradients before accumulating
        optimizer.zero_grad()

        # Start index of the same effective batch of 64
        batch_start = (step * BATCH_SIZE) % NUM_SAMPLES

        effective_batch_loss = 0.0

        # --------------------------------------------------
        # Divide the effective batch into 4 micro-batches
        # --------------------------------------------------

        for accumulation_step in range(ACCUMULATION_STEPS):

            start = (
                batch_start
                + accumulation_step * MICRO_BATCH_SIZE
            )

            end = start + MICRO_BATCH_SIZE

            # Select micro-batch from CPU dataset
            batch_X = X[start:end].to(device)
            batch_y = y[start:end].to(device)

            # Forward pass
            outputs = model(batch_X)

            # Calculate the normal loss
            raw_loss = criterion(outputs, batch_y)

            # Save loss for reporting
            effective_batch_loss += raw_loss.item()

            # Divide loss so accumulated gradient represents
            # the average gradient of a batch of 64
            loss = raw_loss / ACCUMULATION_STEPS

            # Accumulate gradients
            loss.backward()

        # Update weights only after all 4 micro-batches
        optimizer.step()

    # Wait for GPU operations to finish
    torch.cuda.synchronize()

    training_time = time.perf_counter() - start_time

    # Measure peak GPU memory
    peak_memory = (
        torch.cuda.max_memory_allocated()
        / (1024 ** 2)
    )

    # Average loss across the final effective batch of 64
    final_loss = (
        effective_batch_loss / ACCUMULATION_STEPS
    )

    # Store results
    accum_times.append(training_time)
    accum_memories.append(peak_memory)
    accum_losses.append(final_loss)

    # Free GPU memory before next repetition
    del model
    torch.cuda.empty_cache()


# --------------------------------------------------
# Calculate mean results
# --------------------------------------------------

accum_time_mean = np.mean(accum_times)
accum_memory_mean = np.mean(accum_memories)
accum_loss_mean = np.mean(accum_losses)


# --------------------------------------------------
# Display results
# --------------------------------------------------

print("Gradient Accumulation Results")
print("-----------------------------")

print("Micro-batch size:", MICRO_BATCH_SIZE)
print("Accumulation steps:", ACCUMULATION_STEPS)
print("Effective batch size:",
      MICRO_BATCH_SIZE * ACCUMULATION_STEPS)

print(f"\nMean Training Time: {accum_time_mean:.4f} seconds")
print(f"Mean Peak GPU Memory: {accum_memory_mean:.2f} MB")
print(f"Mean Final Loss: {accum_loss_mean:.4f}")

Gradient Accumulation Results
-----------------------------
Micro-batch size: 16
Accumulation steps: 4
Effective batch size: 64

Mean Training Time: 0.4128 seconds
Mean Peak GPU Memory: 19.06 MB
Mean Final Loss: 2.2893


In [14]:
# Q3 - Step 8: Gradient Accumulation experiment

REPEATS = 3

# --------------------------------------------------
# Gradient accumulation settings
#
# Effective batch size stays 64:
# 16 samples × 4 accumulation steps = 64 samples
# --------------------------------------------------

MICRO_BATCH_SIZE = 16
ACCUMULATION_STEPS = BATCH_SIZE // MICRO_BATCH_SIZE

accum_times = []
accum_memories = []
accum_losses = []


# --------------------------------------------------
# Run the experiment 3 times
# --------------------------------------------------

for repeat in range(REPEATS):

    # Use reproducible seeds
    current_seed = SEED + repeat

    random.seed(current_seed)
    np.random.seed(current_seed)
    torch.manual_seed(current_seed)
    torch.cuda.manual_seed_all(current_seed)

    # Create the same model architecture
    model = SimpleNet().to(device)

    # Same loss function and optimizer as baseline
    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001
    )

    # Reset peak GPU memory measurement
    torch.cuda.reset_peak_memory_stats()

    # Synchronize GPU before timing
    torch.cuda.synchronize()
    start_time = time.perf_counter()

    model.train()

    # --------------------------------------------------
    # Keep the same 100 optimizer training steps
    # --------------------------------------------------

    for step in range(TRAINING_STEPS):

        # Clear gradients before accumulating
        optimizer.zero_grad()

        # Start index of the same effective batch of 64
        batch_start = (step * BATCH_SIZE) % NUM_SAMPLES

        effective_batch_loss = 0.0

        # --------------------------------------------------
        # Divide the effective batch into 4 micro-batches
        # --------------------------------------------------

        for accumulation_step in range(ACCUMULATION_STEPS):

            start = (
                batch_start
                + accumulation_step * MICRO_BATCH_SIZE
            )

            end = start + MICRO_BATCH_SIZE

            # Select micro-batch from CPU dataset
            batch_X = X[start:end].to(device)
            batch_y = y[start:end].to(device)

            # Forward pass
            outputs = model(batch_X)

            # Calculate the normal loss
            raw_loss = criterion(outputs, batch_y)

            # Save loss for reporting
            effective_batch_loss += raw_loss.item()

            # Divide loss so accumulated gradient represents
            # the average gradient of a batch of 64
            loss = raw_loss / ACCUMULATION_STEPS

            # Accumulate gradients
            loss.backward()

        # Update weights only after all 4 micro-batches
        optimizer.step()

    # Wait for GPU operations to finish
    torch.cuda.synchronize()

    training_time = time.perf_counter() - start_time

    # Measure peak GPU memory
    peak_memory = (
        torch.cuda.max_memory_allocated()
        / (1024 ** 2)
    )

    # Average loss across the final effective batch of 64
    final_loss = (
        effective_batch_loss / ACCUMULATION_STEPS
    )

    # Store results
    accum_times.append(training_time)
    accum_memories.append(peak_memory)
    accum_losses.append(final_loss)

    # Free GPU memory before next repetition
    del model
    torch.cuda.empty_cache()


# --------------------------------------------------
# Calculate mean results
# --------------------------------------------------

accum_time_mean = np.mean(accum_times)
accum_memory_mean = np.mean(accum_memories)
accum_loss_mean = np.mean(accum_losses)


# --------------------------------------------------
# Display results
# --------------------------------------------------

print("Gradient Accumulation Results")
print("-----------------------------")

print("Micro-batch size:", MICRO_BATCH_SIZE)
print("Accumulation steps:", ACCUMULATION_STEPS)
print("Effective batch size:",
      MICRO_BATCH_SIZE * ACCUMULATION_STEPS)

print(f"\nMean Training Time: {accum_time_mean:.4f} seconds")
print(f"Mean Peak GPU Memory: {accum_memory_mean:.2f} MB")
print(f"Mean Final Loss: {accum_loss_mean:.4f}")

Gradient Accumulation Results
-----------------------------
Micro-batch size: 16
Accumulation steps: 4
Effective batch size: 64

Mean Training Time: 0.4048 seconds
Mean Peak GPU Memory: 19.06 MB
Mean Final Loss: 2.2893


In [15]:
# Q3 - Step 9: Mixed Precision Training experiment

REPEATS = 3

mixed_times = []
mixed_memories = []
mixed_losses = []


# --------------------------------------------------
# Run the mixed precision experiment 3 times
# --------------------------------------------------

for repeat in range(REPEATS):

    # Use reproducible seeds
    current_seed = SEED + repeat

    random.seed(current_seed)
    np.random.seed(current_seed)
    torch.manual_seed(current_seed)
    torch.cuda.manual_seed_all(current_seed)

    # Create the same neural network
    model = SimpleNet().to(device)

    # Same loss function as baseline
    criterion = nn.CrossEntropyLoss()

    # Same optimizer and learning rate
    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001
    )

    # --------------------------------------------------
    # Gradient scaler
    # Helps prevent numerical underflow when using FP16
    # --------------------------------------------------

    scaler = torch.amp.GradScaler("cuda")

    # Reset GPU peak-memory measurement
    torch.cuda.reset_peak_memory_stats()

    # Synchronize before starting the timer
    torch.cuda.synchronize()
    start_time = time.perf_counter()

    model.train()

    # --------------------------------------------------
    # Same 100 training steps as baseline
    # --------------------------------------------------

    for step in range(TRAINING_STEPS):

        start = (step * BATCH_SIZE) % NUM_SAMPLES
        end = start + BATCH_SIZE

        # Select the same batch from the CPU dataset
        batch_X = X[start:end].to(device)
        batch_y = y[start:end].to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # --------------------------------------------------
        # Mixed precision forward pass
        # Suitable operations are performed using FP16
        # --------------------------------------------------

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            outputs = model(batch_X)

            loss = criterion(
                outputs,
                batch_y
            )

        # --------------------------------------------------
        # Scale loss before backpropagation
        # --------------------------------------------------

        scaler.scale(loss).backward()

        # Update model weights safely
        scaler.step(optimizer)

        # Update scaling factor
        scaler.update()


    # Wait for all GPU operations to finish
    torch.cuda.synchronize()

    training_time = (
        time.perf_counter() - start_time
    )


    # --------------------------------------------------
    # Measure peak GPU memory
    # --------------------------------------------------

    peak_memory = (
        torch.cuda.max_memory_allocated()
        / (1024 ** 2)
    )


    # --------------------------------------------------
    # Store experiment results
    # --------------------------------------------------

    mixed_times.append(training_time)
    mixed_memories.append(peak_memory)
    mixed_losses.append(loss.item())


    # Free GPU memory before next repetition
    del model
    torch.cuda.empty_cache()


# --------------------------------------------------
# Calculate mean results
# --------------------------------------------------

mixed_time_mean = np.mean(mixed_times)
mixed_memory_mean = np.mean(mixed_memories)
mixed_loss_mean = np.mean(mixed_losses)


# --------------------------------------------------
# Display results
# --------------------------------------------------

print("Mixed Precision Results")
print("-----------------------")

print(f"Mean Training Time: {mixed_time_mean:.4f} seconds")
print(f"Mean Peak GPU Memory: {mixed_memory_mean:.2f} MB")
print(f"Mean Final Loss: {mixed_loss_mean:.4f}")

Mixed Precision Results
-----------------------
Mean Training Time: 0.2974 seconds
Mean Peak GPU Memory: 19.35 MB
Mean Final Loss: 2.2894


In [16]:
# Q3 - Step 10: Create final comparison table

# --------------------------------------------------
# Build a table using the results from all experiments
# --------------------------------------------------

results = pd.DataFrame({

    "Experiment": [
        "Baseline",
        "Weight Initialization - Default",
        "Weight Initialization - Xavier",
        "Activation Checkpointing",
        "Gradient Accumulation",
        "Mixed Precision"
    ],

    "Training Time (s)": [
        baseline_time,
        default_time_mean,
        xavier_time_mean,
        checkpoint_time_mean,
        accum_time_mean,
        mixed_time_mean
    ],

    "Peak GPU Memory (MB)": [
        baseline_memory,
        default_memory_mean,
        xavier_memory_mean,
        checkpoint_memory_mean,
        accum_memory_mean,
        mixed_memory_mean
    ],

    "Final Loss": [
        baseline_loss,
        default_loss_mean,
        xavier_loss_mean,
        checkpoint_loss_mean,
        accum_loss_mean,
        mixed_loss_mean
    ]
})


# --------------------------------------------------
# Round values to make the table easier to read
# --------------------------------------------------

results["Training Time (s)"] = (
    results["Training Time (s)"].round(4)
)

results["Peak GPU Memory (MB)"] = (
    results["Peak GPU Memory (MB)"].round(2)
)

results["Final Loss"] = (
    results["Final Loss"].round(4)
)


# --------------------------------------------------
# Display final comparison table
# --------------------------------------------------

print("Q3 Training Optimization Comparison")
print("-----------------------------------")

display(results)


# --------------------------------------------------
# Display tensor creation benchmark separately
# This is a tensor-operation benchmark rather than
# a complete model-training experiment.
# --------------------------------------------------

tensor_results = pd.DataFrame({

    "Tensor Creation Method": [
        "CPU Creation + GPU Transfer",
        "Direct GPU Creation"
    ],

    "Mean Time (s)": [
        cpu_to_gpu_mean,
        direct_gpu_mean
    ]
})

tensor_results["Mean Time (s)"] = (
    tensor_results["Mean Time (s)"].round(4)
)

print("\nTensor Creation Comparison")
print("--------------------------")

display(tensor_results)

print(
    f"\nDirect GPU tensor creation speedup: "
    f"{speedup:.2f}x"
)

Q3 Training Optimization Comparison
-----------------------------------


,Experiment,Training Time (s),Peak GPU Memory (MB),Final Loss
0,Baseline,0.1287,18.03,2.2875
1,Weight Initialization - Default,0.1221,18.06,2.2893
2,Weight Initialization - Xavier,0.1215,18.06,2.4267
3,Activation Checkpointing,0.2367,19.08,2.2893
4,Gradient Accumulation,0.4048,19.06,2.2893
5,Mixed Precision,0.2974,19.35,2.2894



Tensor Creation Comparison
--------------------------


,Tensor Creation Method,Mean Time (s)
0,CPU Creation + GPU Transfer,0.0786
1,Direct GPU Creation,0.0106



Direct GPU tensor creation speedup: 7.39x


In [17]:
# Q3 - Step 11: Repeat baseline experiment for fair comparison

REPEATS = 3

baseline_times = []
baseline_memories = []
baseline_losses = []


# --------------------------------------------------
# Run baseline training 3 times
# --------------------------------------------------

for repeat in range(REPEATS):

    # Use reproducible seed for each repetition
    current_seed = SEED + repeat

    random.seed(current_seed)
    np.random.seed(current_seed)
    torch.manual_seed(current_seed)
    torch.cuda.manual_seed_all(current_seed)

    # Create model using default initialization
    model = SimpleNet()

    # Run the same baseline training procedure
    training_time, peak_memory, final_loss = train_model(
        model,
        X,
        y
    )

    # Save measurements
    baseline_times.append(training_time)
    baseline_memories.append(peak_memory)
    baseline_losses.append(final_loss)

    # Free GPU memory before next repetition
    del model
    torch.cuda.empty_cache()


# --------------------------------------------------
# Calculate mean baseline measurements
# --------------------------------------------------

baseline_time_mean = np.mean(baseline_times)
baseline_memory_mean = np.mean(baseline_memories)
baseline_loss_mean = np.mean(baseline_losses)


# --------------------------------------------------
# Display final baseline results
# --------------------------------------------------

print("Repeated Baseline Results")
print("-------------------------")

print(f"Mean Training Time: {baseline_time_mean:.4f} seconds")
print(f"Mean Peak GPU Memory: {baseline_memory_mean:.2f} MB")
print(f"Mean Final Loss: {baseline_loss_mean:.4f}")

Repeated Baseline Results
-------------------------
Mean Training Time: 0.1212 seconds
Mean Peak GPU Memory: 19.93 MB
Mean Final Loss: 2.2893


In [18]:
# Q3 - Step 12: Create final corrected comparison table

# --------------------------------------------------
# Use repeated mean measurements for all experiments
# --------------------------------------------------

final_results = pd.DataFrame({

    "Experiment": [
        "Baseline",
        "Weight Initialization - Default",
        "Weight Initialization - Xavier",
        "Activation Checkpointing",
        "Gradient Accumulation",
        "Mixed Precision"
    ],

    "Mean Training Time (s)": [
        baseline_time_mean,
        default_time_mean,
        xavier_time_mean,
        checkpoint_time_mean,
        accum_time_mean,
        mixed_time_mean
    ],

    "Mean Peak GPU Memory (MB)": [
        baseline_memory_mean,
        default_memory_mean,
        xavier_memory_mean,
        checkpoint_memory_mean,
        accum_memory_mean,
        mixed_memory_mean
    ],

    "Mean Final Loss": [
        baseline_loss_mean,
        default_loss_mean,
        xavier_loss_mean,
        checkpoint_loss_mean,
        accum_loss_mean,
        mixed_loss_mean
    ]
})


# --------------------------------------------------
# Round values for clean presentation
# --------------------------------------------------

final_results["Mean Training Time (s)"] = (
    final_results["Mean Training Time (s)"].round(4)
)

final_results["Mean Peak GPU Memory (MB)"] = (
    final_results["Mean Peak GPU Memory (MB)"].round(2)
)

final_results["Mean Final Loss"] = (
    final_results["Mean Final Loss"].round(4)
)


# --------------------------------------------------
# Display final training comparison
# --------------------------------------------------

print("Final Q3 Training Optimization Comparison")
print("-----------------------------------------")

display(final_results)


# --------------------------------------------------
# Tensor creation results
# --------------------------------------------------

tensor_results = pd.DataFrame({

    "Tensor Creation Method": [
        "CPU Creation + GPU Transfer",
        "Direct GPU Creation"
    ],

    "Mean Time (s)": [
        cpu_to_gpu_mean,
        direct_gpu_mean
    ]
})

tensor_results["Mean Time (s)"] = (
    tensor_results["Mean Time (s)"].round(4)
)

print("\nTensor Creation Comparison")
print("--------------------------")

display(tensor_results)

print(
    f"\nDirect GPU tensor creation speedup: "
    f"{speedup:.2f}x"
)

Final Q3 Training Optimization Comparison
-----------------------------------------


,Experiment,Mean Training Time (s),Mean Peak GPU Memory (MB),Mean Final Loss
0,Baseline,0.1212,19.93,2.2893
1,Weight Initialization - Default,0.1221,18.06,2.2893
2,Weight Initialization - Xavier,0.1215,18.06,2.4267
3,Activation Checkpointing,0.2367,19.08,2.2893
4,Gradient Accumulation,0.4048,19.06,2.2893
5,Mixed Precision,0.2974,19.35,2.2894



Tensor Creation Comparison
--------------------------


,Tensor Creation Method,Mean Time (s)
0,CPU Creation + GPU Transfer,0.0786
1,Direct GPU Creation,0.0106



Direct GPU tensor creation speedup: 7.39x


The model, dataset, batch size, and training procedures were all the same in the optimization experiments. With a speedup of 7.39×, direct GPU tensor creation demonstrated the greatest improvement, taking 0.0106 s as opposed to 0.0786 s for CPU creation followed by GPU transfer. The time and memory consumption of the default and Xavier initialization were comparable, but Xavier resulted in a greater final loss (2.4267 vs. 2.2893). In this experiment, activation checkpointing, gradient accumulation, and mixed precision all kept the final loss near the baseline but lengthened the training period and did not lower peak GPU memory. This is primarily due to the model's small size, which makes the overhead of these methods greater than their advantages. Although direct GPU tensor creation was the most successful optimization in this experiment, these optimizations are generally more beneficial for larger models.